# Data wrangling with Pandas: statistics and reshaping

Lino Galiana  
2026-07-20

<div class="badge-container"><div class="badge-text">If you want to try the examples in this tutorial:</div><a href="https://github.com/linogaliana/python-datascientist-notebooks/blob/main/notebooks/en/manipulation/02_pandas_stats.ipynb" class="badge" target="_blank" rel="noopener"><img src="https://img.shields.io/static/v1?logo=github&label=&message=View%20on%20GitHub&color=181717" alt="View on GitHub"></a>
<a href="https://datalab.sspcloud.fr/launcher/ide/vscode-python?autoLaunch=true&name=«02_pandas_stats»&init.personalInit=«https%3A%2F%2Fraw.githubusercontent.com%2Flinogaliana%2Fpython-datascientist%2Fmain%2Fsspcloud%2Finit-vscode.sh»&init.personalInitArgs=«en/manipulation%2002_pandas_stats»" class="badge" target="_blank" rel="noopener"><img src="https://custom-icon-badges.demolab.com/badge/SSP%20Cloud-Lancer_avec_VSCode-blue?logo=vsc&logoColor=white" alt="Onyxia"></a>
<a href="https://datalab.sspcloud.fr/launcher/ide/jupyter-python?autoLaunch=true&name=«02_pandas_stats»&init.personalInit=«https%3A%2F%2Fraw.githubusercontent.com%2Flinogaliana%2Fpython-datascientist%2Fmain%2Fsspcloud%2Finit-jupyter.sh»&init.personalInitArgs=«en/manipulation%2002_pandas_stats»" class="badge" target="_blank" rel="noopener"><img src="https://img.shields.io/badge/SSP%20Cloud-Lancer_avec_Jupyter-orange?logo=Jupyter&logoColor=orange" alt="Onyxia"></a>
<a href="https://colab.research.google.com/github/linogaliana/python-datascientist-notebooks-colab//en/blob/main//notebooks/en/manipulation/02_pandas_stats.ipynb" class="badge" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>
<a href="https://codespaces.new/linogaliana/python-datascientist-notebooks?quickstart=1&ref=main" class="badge" target="_blank" rel="noopener"><img src="https://github.com/codespaces/badge.svg" alt="Open in GitHub Codespaces"></a><br></div>

> **None**
>
> <div class="callout callout-style-default callout-note callout-titled">
> <div class="callout-header d-flex align-content-center">
> <div class="callout-icon-container">
> <i class="callout-icon"></i>
> </div>
> <div class="callout-title-container flex-fill">
> Note
> </div>
> </div>
> <div class="callout-body-container callout-body">
>
> Ceci est la version française 🇫🇷 de ce chapitre, pour voir la version anglaise rendez-vous sur <a href="https://pythonds.linogaliana.fr//home/runner/work/python-datascientist/python-datascientist/en/content/manipulation/02_pandas_stats.qmd">le site du cours</a>.
>
> </div>
> </div>

> **Tip**
>
> <div class="callout callout-style-default callout-tip callout-titled">
> <div class="callout-header d-flex align-content-center">
> <div class="callout-icon-container">
> <i class="callout-icon"></i>
> </div>
> <div class="callout-title-container flex-fill">
> Skills to be acquired by the end of this chapter
> </div>
> </div>
> <div class="callout-body-container callout-body">
>
> -   How to construct fine aggregate statistics using `Pandas` methods;
> -   Know how to reshape your data from *long* to *wide* format (and vice versa);
> -   Create attractive tables to communicate aggregated results.
>
> </div>
> </div>

# 1. Introduction

The [previous chapter](../../content/manipulation/02_pandas_joins.qmd) showed how to enrich a dataset by associating it with another source through a join. We now have two datasets that have been made consistent with one another: the Ademe greenhouse gas emissions data (`emissions`) and the Insee Filosofi framing data (`filosofi`).

This chapter will use this association to deepen our understanding of the data through two complementary operations:

-   group descriptive statistics;
-   reshaping data between *long* and *wide* formats.

Finally, we will see how to build, from these statistics, polished communication tables using the `great_tables` package.

## 1.1 Data

We start again from the same datasets as in the [chapter on joins](../../content/manipulation/02_pandas_joins.qmd): the Ademe municipal emissions data (`emissions`) and the Insee Filosofi data (`filosofi`), enriched with one another. The code below, already explained in detail in the previous chapter, retrieves these datasets again:

In [ ]:
import requests
import pandas as pd

def get_melodi(dataset, dimension, code, value_name, time_period, geo_level="COM"):

  url = f"https://api.insee.fr/melodi/data/{dataset}"

  params = {
    "GEO": geo_level,
    dimension: code,
    "TIME_PERIOD": time_period,
    "maxResult": 40000,
  }

  response = requests.get(url, params=params, timeout=60)
  response.raise_for_status()
  observations = response.json()["observations"]

  return pd.DataFrame(
    {
      "CODGEO": obs["dimensions"]["GEO"].split("-")[-1],
      value_name: obs["measures"]["OBS_VALUE_NIVEAU"].get("value"),
    }
    for obs in observations
  )

niveau_vie = get_melodi("DS_FILOSOFI_CC", "FILOSOFI_MEASURE", "MED_SL", "NIVVIE_MEDIAN", 2023)
taux_pauvrete = get_melodi("DS_FILOSOFI_CC", "FILOSOFI_MEASURE", "PR_MD60", "TAUX_PAUVRETE", 2023)
population = get_melodi("DS_POPULATIONS_HISTORIQUES", "POPREF_MEASURE", "PMUN", "POPULATION", 2023)

url_cog_2023 = "https://www.insee.fr/fr/statistiques/fichier/6800675/v_commune_2023.csv"
url_backup = "https://minio.lab.sspcloud.fr/lgaliana/data/python-ENSAE/cog_2023.csv"

try:
  cog_2023 = pd.read_csv(url_cog_2023)
except requests.exceptions.Timeout:
  cog_2023 = pd.read_csv(url_backup)

noms_communes = (
  cog_2023
  .loc[cog_2023["TYPECOM"] == "COM", ["COM", "LIBELLE"]]
  .rename(columns={"COM": "CODGEO", "LIBELLE": "LIBGEO"})
)

filosofi = (
  niveau_vie
  .merge(taux_pauvrete, on="CODGEO", how="outer")
  .merge(population, on="CODGEO", how="outer")
  .merge(noms_communes, on="CODGEO", how="left")
  [["CODGEO", "LIBGEO", "POPULATION", "NIVVIE_MEDIAN", "TAUX_PAUVRETE"]]
)

filosofi["POPULATION"] = filosofi["POPULATION"].astype(int)
filosofi["dep"] = filosofi["CODGEO"].str[:2]

In [ ]:
import pandas as pd

url = "https://data.ademe.fr/data-fair/api/v1/datasets/igt-pouvoir-de-rechauffement-global/convert"
emissions = pd.read_csv(url)
emissions["dep"] = emissions["INSEE commune"].str[:2]

emissions.head(2)

We will also need the following *packages*

In [ ]:
import numpy as np

To obtain reproducible results, you can set the seed of the pseudo-random number generator.

In [ ]:
np.random.seed(123)

# 2. Descriptive statistics by group

## 2.1 Principle

In the introductory chapter, we saw how to obtain an aggregated statistic easily with `Pandas`. However, it is common to have data with intermediate analysis strata that are relevant: geographical variables, membership in socio-demographic groups related to recorded characteristics, temporal period indicators, etc. To better understand the structure of the data, data scientists are often led to construct descriptive statistics on sub-groups present in the data.

For example, we previously constructed emission statistics at the national level. But what about the emission profiles of different departments? To answer this question, it will be useful to aggregate our data at the departmental level. This will give us different information from the initial dataset (municipal level) and the most aggregated level (national level).

In `SQL`, it is very simple to segment data to perform operations on coherent blocks and recollect results in the appropriate dimension. The underlying logic is that of *split-apply-combine*, which is adopted by data manipulation languages, including `pandas` [which is no exception](https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html).

The following image, from [this site](https://unlhcc.github.io/r-novice-gapminder/16-plyr/), well represents how the `split`-`apply`-`combine` approach works:

<figure>
<img src="https://unlhcc.github.io/r-novice-gapminder/fig/12-plyr-fig1.png" alt="Split-apply-combine (Source: unlhcc.github.io)" />
<figcaption aria-hidden="true">Split-apply-combine (Source: <a href="https://unlhcc.github.io/r-novice-gapminder/16-plyr/">unlhcc.github.io</a>)</figcaption>
</figure>

In `Pandas`, we use `groupby` to segment the data according to one or more axes (this [tutorial](https://realpython.com/pandas-groupby/) on the subject is particularly useful). All the aggregation operations (counting, averages, etc.) that we saw earlier can be applied by group.

Technically, this operation involves creating an association between labels (values of group variables) and observations. Using the `groupby` method does not trigger operations until a statistic is implemented; it simply creates a formal relationship between observations and groupings that will be used later:

In [ ]:
filosofi.groupby('dep').__class__

pandas.api.typing.DataFrameGroupBy

As long as we do not call an action on a `DataFrame` by group, such as `head` or `display`, `pandas` performs no operations. This is called *lazy evaluation*. For example, the result of `df.groupby('dep')` is a transformation that has not yet been evaluated:

In [ ]:
filosofi.groupby('dep')

## 2.2 Illustration 1: counting by group

To illustrate the application of this principle to counting, we can count the number of municipalities by department in 2023 (this statistic changes every year due to municipal mergers). For this, we simply take the reference of French municipalities from the official geographical code (COG) and count by department using `count`:

With this dataset, without resorting to group statistics, we can already know how many municipalities, departments, and regions we have in France, respectively:

In [ ]:
communes = cog_2023.loc[cog_2023['TYPECOM']=="COM"]
communes.loc[:, ['COM', 'DEP', 'REG']].nunique()

COM    34945
DEP      101
REG       18
dtype: int64

Now, let’s look at the departments with the most municipalities. It is the same counting function where we play, this time, on the group from which the statistic is calculated.

Calculating this statistic is quite straightforward when you understand the principle of calculating statistics with `Pandas`:

In [ ]:
communes = cog_2023.loc[cog_2023['TYPECOM']=="COM"]
communes.groupby('DEP').agg({'COM': 'nunique'})

101 rows × 1 columns

In SQL, we would use the following query:

``` sql
SELECT dep, COUNT DISTINCT "COM" AS COM
FROM communes
GROUP BY dep
WHERE TYPECOM == 'COM';
```

The output is an indexed `Series`. This is not very convenient as we mentioned in the previous chapter. It is more practical to transform this object into a `DataFrame` with `reset_index`. Finally, with `sort_values`, we obtain the desired statistic:

In [ ]:
(
    communes
    .groupby('DEP')
    .agg({'COM': 'nunique'})
    .reset_index()
    .sort_values('COM', ascending = False)
)

101 rows × 2 columns

## 2.3 Illustration 2: aggregates by group

To illustrate aggregates by group, we can use the Insee `filosofi` dataset and sum the `POPULATION` variable.

To calculate the total for the whole of France, we can do it in two ways:

In [ ]:
filosofi['POPULATION'].sum()* 1e-6

np.float64(68.09428)

In [ ]:
filosofi.agg({"POPULATION": "sum"}).div(1e6)

POPULATION    68.09428
dtype: float64

where the results are reported in millions of people. The logic is the same when doing group statistics, it’s just a matter of replacing `filosofi` with `filosofi.groupby('dep')` to create a partitioned version of our dataset by department:

In [ ]:
filosofi.groupby('dep')['POPULATION'].sum()

dep
01     679344
02     523342
03     333298
04     168054
05     143467
       ...   
92    1654712
93    1704316
94    1426929
95    1281653
97    1928465
Name: POPULATION, Length: 97, dtype: int64

In [ ]:
filosofi.groupby('dep').agg({"POPULATION": "sum"})

97 rows × 1 columns

The second approach is more practical because it directly gives a `Pandas` `DataFrame` and not an unnamed indexed series. From this, a few basic manipulations can suffice to have a shareable table on departmental demographics. However, this table would be somewhat rudimentary as we currently only have the department numbers. To get the names of the departments, we would need to use a second dataset and merge the common information between them (in this case, the department number): this is exactly the join principle we implemented in the [previous chapter](../../content/manipulation/02_pandas_joins.qmd).

# 3. Application

This application exercise uses the `Ademe` dataset named `emissions` previously discussed.

In [ ]:
secteurs = emissions.select_dtypes(include='number').columns

> **Tip**
>
> <div class="callout callout-style-default callout-tip callout-titled">
> <div class="callout-header d-flex align-content-center">
> <div class="callout-icon-container">
> <i class="callout-icon"></i>
> </div>
> <div class="callout-title-container flex-fill">
> Exercise 1: Group Aggregations
> </div>
> </div>
> <div class="callout-body-container callout-body">
>
> 1.  Calculate the total emissions of the “Residential” sector by department and compare the value to the most polluting department in this sector. Draw insights from the reality that this statistic reflects.
>
> 2.  Calculate the total emissions for each sector in each department. For each department, calculate the proportion of total emissions coming from each sector.
>
> <details>
>
> <summary>
>
> Hint for this question
>
> </summary>
>
> -   *“Group by”* = `groupby`
> -   *“Total emissions”* = `agg({*** : "sum"})`
>
> </details>
>
> </div>
> </div>

In question 1, the result should be as follows:

In [ ]:
# Question 1
emissions_residentielles = (
    emissions
    .groupby("dep")
    .agg({"Résidentiel" : "sum"})
    .reset_index()
    .sort_values("Résidentiel", ascending = False)
)
emissions_residentielles["Résidentiel (% valeur max)"] = emissions_residentielles["Résidentiel"]/emissions_residentielles["Résidentiel"].max()
emissions_residentielles.head(5)

This ranking may reflect demographics rather than the process we wish to measure. Without the addition of information on the population of each département to control for this factor, it is difficult to know whether there is a structural difference in behavior between the inhabitants of Nord (département 59) and Moselle (département 57).

In [ ]:
# Question 2
emissions_par_departement = (
    emissions.groupby('dep').sum(numeric_only=True)
)
emissions_par_departement['total'] = emissions_par_departement.sum(axis = 1)
emissions_par_departement["Part " + secteurs] = (
    emissions_par_departement
    .loc[:, secteurs]
    .div(emissions_par_departement['total'], axis = 0)
    .mul(100)
)

At the end of question 2, let’s take the share of emissions from agriculture and the tertiary sector in departmental emissions:

In [ ]:
emissions_par_departement.sort_values("Part Agriculture", ascending = False).head(5)

5 rows × 21 columns

In [ ]:
emissions_par_departement.sort_values("Part Tertiaire", ascending = False).head(5)

5 rows × 21 columns

These results are quite logical; rural departments have a larger share of their emissions from agriculture, while urban departments have higher emissions from the tertiary sector, which is related to the higher density of these areas.

With these statistics, we progress in understanding our dataset and, consequently, the nature of CO2 emissions in France. Descriptive statistics by group help us better grasp the spatial heterogeneity of our phenomenon.

However, we would remain limited in our ability to interpret these statistics without additional information: a département could look like a low emitter simply because it is sparsely populated. This is exactly the kind of limitation we resolved thanks to enriching our data with a join against the Filosofi data (see in particular the computation of a carbon footprint per capita in the previous chapter).

# 4. Restructuring datasets

## 4.1 Principle

When we have multiple pieces of information for the same individual or group, we generally find two types of data structures:

-   **Wide** format: the data contains repeated observations for the same individual (or group) in different columns.
-   **Long** format: the data contains repeated observations for the same individual in different rows, with a column distinguishing the observation levels.

An example of the distinction between the two can be taken from Hadley Wickham’s reference book, [*R for Data Science*](https://r4ds.hadley.nz/):

<figure>
<img src="https://d33wubrfki0l68.cloudfront.net/3aea19108d39606bbe49981acda07696c0c7fcd8/2de65/images/tidy-9.png" alt="Wide and Long Data Formats (Source: R for Data Science)" />
<figcaption aria-hidden="true">Wide and Long Data Formats (Source: <a href="https://r4ds.hadley.nz/"><em>R for Data Science</em></a>)</figcaption>
</figure>

The following cheat sheet will help remember the functions to apply if needed:

![](https://minio.lab.sspcloud.fr/lgaliana/generative-art/pythonds/reshape.png)

Switching from a *wide* format to a *long* format (or vice versa) can be extremely practical because certain functions are more suitable for one form of data than the other.

Generally, with `Python` as with `R`, **long formats are often preferable**. Wide formats are rather designed for spreadsheets like `Excel`, where we have a limited number of rows to create pivot tables from.

## 4.2 Application

The ADEME data, and the Insee data as well, are in the *wide* format. The next exercise illustrates the benefit of converting from *long* to *wide* before creating a plot with the `plot` method seen in the introductory chapter.

> **Tip**
>
> <div class="callout callout-style-default callout-tip callout-titled">
> <div class="callout-header d-flex align-content-center">
> <div class="callout-icon-container">
> <i class="callout-icon"></i>
> </div>
> <div class="callout-title-container flex-fill">
> Exercice 2: Restructuring Data: Wide to Long
> </div>
> </div>
> <div class="callout-body-container callout-body">
>
> 1.  Create a copy of the ADEME data by doing `df_wide = emissions.copy()`
>
> 2.  Restructure the data into the *long* format to have emission data by sector while keeping the commune as the level of analysis (pay attention to other identifying variables).
>
> 3.  Sum the emissions by sector and represent it graphically.
>
> 4.  For each department, identify the most polluting sector.
>
> </div>
> </div>

In [ ]:
# Question 1

emissions_wide = emissions.copy()
emissions_wide[['Commune','dep', "Agriculture", "Tertiaire"]].head() 

In [ ]:
# Question 2
emissions_wide.reset_index().melt(id_vars = ['INSEE commune','Commune','dep'],
                          var_name = "secteur", value_name = "emissions")

In [ ]:
# Question 3

emissions_totales = (
  emissions_wide.reset_index(drop=True)
 .melt(
    id_vars = ['INSEE commune','Commune','dep'],
    var_name = "secteur", value_name = "emissions"
    )
 .groupby('secteur')
 .sum(numeric_only = True)
)

emissions_totales.plot(kind = "barh")

In [ ]:
# Question 4

top_commune_dep = (
  emissions_wide
  .reset_index(drop=True)
  .melt(
    id_vars = ['INSEE commune','Commune','dep'],
    var_name = "secteur", value_name = "emissions"
  )
 .groupby(['secteur','dep'])
 .sum(numeric_only=True).reset_index()
 .sort_values(['dep','emissions'], ascending = False)
 .groupby('dep').head(1)
)
display(top_commune_dep)

# 5. Formatting descriptive statistics tables

A `Pandas` DataFrame is automatically formatted when viewed from a notebook as a minimally styled HTML table. This formatting is convenient for viewing data, a necessary task for data scientists, but it doesn’t go much beyond that.

In an exploratory phase, it can be useful to have a more complete table, including minimal visualizations, to better understand the data. In the final phase of a project, when communicating about it, having an attractive visualization is advantageous. The outputs of notebooks are not a satisfactory solution for these needs and require the medium of the notebook, which can deter some users.

Fortunately, the young package [`great_tables`](https://posit-dev.github.io/great-tables/get-started/) allows for the creation of tables programmatically that rival tedious manual productions in `Excel` and are difficult to replicate. This package is a `Python` port of the `GT` package. `great_tables` builds *HTML* tables, offering great formatting richness and excellent integration with [`Quarto`](https://quarto.org/), the reproducible publishing tool developed by RStudio.

The following exercise will propose building a table with this package, step by step. It requires installing the `great_tables` package beforehand:

In [ ]:
!pip install great_tables --quiet

To focus on table construction, the necessary data preparations are provided directly. We will start from the `emissions_merged` dataset, built in the previous chapter when computing the carbon footprint per capita, which looks like this:

To ensure you are able to complete the next exercise, here is the dataframe required for it.

In [ ]:
emissions_totales_commune = emissions.sum(axis = 1, numeric_only = True)

emissions_merged = (
    emissions.assign(emissions = emissions_totales_commune)
    .reset_index()
    .merge(filosofi, left_on = "INSEE commune", right_on = "CODGEO")
)
emissions_merged = emissions_merged.loc[emissions_merged['POPULATION'] > 0]
emissions_merged['empreinte'] = emissions_merged['emissions']/emissions_merged['POPULATION']
emissions_merged['empreinte'] = emissions_merged['empreinte'].astype(float)

In [ ]:
emissions_table = (
    emissions_merged
    .rename(columns={"dep_y": "dep", "POPULATION": "population", "NIVVIE_MEDIAN": "revenu"})
    .groupby("dep")
    .agg({"empreinte": "sum", "revenu": "median", "population": "sum"}) #pas vraiment le revenu médian
    .reset_index()
    .sort_values(by = "empreinte")
)

In this table, we will include horizontal bars, similar to the examples shown [here](https://posit-dev.github.io/great-tables/examples/). This is done by directly including the *HTML* code in the DataFrame column.

In [ ]:
def create_bar(prop_fill: float, max_width: int, height: int, color: str = "green") -> str:
    """Create divs to represent prop_fill as a bar."""
    width = round(max_width * prop_fill, 2)
    px_width = f"{width}px"
    return f"""\
    <div style="width: {max_width}px; background-color: lightgrey;">\
        <div style="height:{height}px;width:{px_width};background-color:{color};"></div>\
    </div>\
    """

colors = {'empreinte': "green", 'revenu': "red", 'population': "blue"}

for variable in ['empreinte', 'revenu', 'population']:
    emissions_table[f'raw_perc_{variable}'] = emissions_table[variable]/emissions_table[variable].max()
    emissions_table[f'bar_{variable}'] = emissions_table[f'raw_perc_{variable}'].map(
        lambda x: create_bar(x, max_width=75, height=20, color = colors[variable])
    )

We keep only the 5 smallest carbon footprints and the five largest.

In [ ]:
emissions_min = emissions_table.head(5).assign(grp = "5 départements les moins pollueurs").reset_index(drop=True)
emissions_max = emissions_table.tail(5).assign(grp = "5 départements les plus pollueurs").reset_index(drop=True)

emissions_table = pd.concat([
    emissions_min,
    emissions_max
])

Finally, to use some practical functions for selecting columns based on patterns, we will convert the data to the [`Polars`](https://pola.rs/) format.

In [ ]:
import polars as pl
emissions_table = pl.from_pandas(emissions_table)

> **Tip**
>
> <div class="callout callout-style-default callout-tip callout-titled">
> <div class="callout-header d-flex align-content-center">
> <div class="callout-icon-container">
> <i class="callout-icon"></i>
> </div>
> <div class="callout-title-container flex-fill">
> Exercise 5: A Beautiful Descriptive Statistics Table (Open Exercise)
> </div>
> </div>
> <div class="callout-body-container callout-body">
>
> Using this base table
>
> ``` python
> GT(emissions_table, groupname_col="grp", rowname_col="dep")
> ```
>
> construct a table in the style of the one below.
>
> </div>
> </div>

In [ ]:
# Start from here
GT(emissions_table, groupname_col="grp", rowname_col="dep")

The table you should have :

In [ ]:
from great_tables import GT, md
import polars.selectors as cs

(
    GT(emissions_table, groupname_col="grp", rowname_col="dep")
    .tab_header(title=md("**Carbon Footprint**"), subtitle = md("_Initial descriptive statistics to refine_"))
    .fmt_number("empreinte", decimals = 2, scale_by=0.1)
    .fmt_number("population", compact=True)
    .fmt_number("revenu", compact=True, decimals = 1)
    .fmt_percent(cs.starts_with("raw_perc"), decimals = 1)
    .cols_move(
        columns=cs.ends_with("_empreinte"),
        after="empreinte"
    )
    .cols_move(
        columns=cs.ends_with("_population"),
        after="population"
    )
    .cols_move(
        columns=cs.ends_with("_revenu"),
        after="revenu"
    )
    .cols_label(
        dep=md("**Department**"),
        empreinte = md("**Carbon Footprint**"),
        revenu = md("**Income**"),
        population = md("**Population**"),
    )
    .cols_label(
        **{
            val: md("_(%)_*") for val in emissions_table.select(cs.contains("raw_perc")).columns
        }
    )    
    .cols_label(
        **{
            val: "" for val in emissions_table.select(cs.contains("bar")).columns
        }
    )    
    .tab_spanner(label=md("**Footprint**"), columns=cs.ends_with("empreinte"))
    .tab_spanner(label=md("**Median Income**"), columns=cs.ends_with("revenu"))
    .tab_spanner(label=md("**Population**"), columns=cs.ends_with("population"))
    .tab_source_note(
        source_note=md("_*__Note__: The median income presented here is an approximation of the department's median income_.")
    )
    .tab_source_note(
        source_note=md("*__Reading__: The __(%)__ columns presented above are scaled to the maximum value of the variable*")
    )
    .tab_source_note(
        source_note=md("*__Source__: Calculations based on Ademe data*")
    )
    .tab_options(
        heading_background_color="#0c540c"
    )

)

Thanks to this, we can already understand that our definition of the carbon footprint is certainly flawed. It seems unlikely that the inhabitants of the 77th department have a carbon footprint 500 times greater than that of intra-muros Paris. The main reason? We are not dealing with a concept of consumption emissions but production emissions, which penalizes industrial areas or areas with airports…

To learn more about constructing tables with `great_tables`, you can replicate this [exercise](https://rgeo.linogaliana.fr/exercises/eval.html) on producing electoral tables that I proposed for an `R` course with `gt`, the equivalent of `great_tables` for `R`.

We have now covered the main features of `Pandas` for manipulating and reshaping data. The [last chapter](../../content/manipulation/02_pandas_beyond.qmd) of this part takes a step back to look at the limitations of `Pandas` syntax and discover the alternative ecosystems that exist to go further.